# Preprocessing 02: Filter and downsample

This notebook filters the eye-synced recording (a band-pass filter: high-pass `HIGHPASS_HZ`, low-pass `LOWPASS_HZ`;
and, if `APPLY_NOTCH` is True, a notch filter at the line-noise frequency from the recording's metadata and its
harmonics) and lowers its sampling rate to `TARGET_SFREQ`. It then does exactly the same to the session's empty-room
recording, so the two can be used together later (for example for the noise covariance in source modelling).
With the notch filter, it also labels the first and last moments of each recording `BAD_edge`, where the filter
can't fully remove the line noise (Step 3c).

**It needs:**

- the eye-synced recording from notebook 05, in `derivatives/eyesync/`;
- the bad channels and breaks from preprocessing notebook 01, in `derivatives/sensorqc/` (for the task recording
  and for the empty room).

**It writes** the filtered, downsampled recordings with their BIDS sidecar files to `derivatives/filtered/`
(Steps 4 and 5). Nothing in the BIDS dataset or in the other derivatives folders is changed.

You only edit Step 1. Then run the cells from top to bottom.


## Step 0: Load the Python packages

This cell loads the tools the notebook uses. You don't need to change anything.

In [ ]:
# --- Standard Python tools (come with Python) ---
from pathlib import Path        # build and check file/folder paths

# --- Scientific Python tools ---
import numpy as np              # numbers and arrays (lists of numbers you can do math on)
import pandas as pd             # tables
import matplotlib.pyplot as plt # plotting
from IPython.display import display   # shows a table nicely in the notebook
from scipy.signal import detrend      # removes a straight-line trend (Step 3c)

# --- MNE-Python: the MEG analysis package ---
import mne

# --- MNE-BIDS: reads BIDS files together with their sidecar files (Step 2) ---
import mne_bids

# Only show MNE's warnings and errors (not every info message)
mne.set_log_level("WARNING")

# Draw plots inside the notebook
%matplotlib inline

print("MNE version:", mne.__version__)
print("MNE-BIDS version:", mne_bids.__version__)

## Step 1: Settings ✏️ (edit this cell)

Everything you normally need to change is in the cell below. 

In [ ]:
# ---------------------------------------------------------------------------
# WHICH RECORDING
# ---------------------------------------------------------------------------
BIDS_ROOT = Path("/jukebox/[yourpath]/auditory-oddball/bids")  # top folder of your BIDS dataset
SUBJECT = "01"           # the part after "sub-"  (as text, in quotes)
SESSION = "01"           # the part after "ses-"; use None (no quotes) if you have no sessions
TASK = "oddballftcued"   # the part after "task-"
RUN = None               # the part after "run-", e.g. "01"; use None if there is no run label
EYESYNC_DERIV = "eyesync"    # the derivatives folder notebook 05 wrote the synced MEG file to
SENSORQC_DERIV = "sensorqc"  # the derivatives folder notebook 01 wrote the bad channels and breaks to (Step 2c)
FILTERED_DERIV = "filtered"  # the derivatives folder this notebook saves the filtered, downsampled recording to (Step 4)

# ---------------------------------------------------------------------------
# FILTER AND DOWNSAMPLE (Step 3)
# ---------------------------------------------------------------------------
HIGHPASS_HZ = 0.1               # remove slow drifts below this frequency (Hz)
TARGET_SFREQ = 600.0            # the new sampling rate (Hz); must be lower than the recording's (Step 2 prints it)
LOWPASS_HZ = TARGET_SFREQ / 3   # removes what can't be represented at the new rate: one third of TARGET_SFREQ,
                                # 200 Hz with the default 600 Hz (see Step 3)
APPLY_NOTCH = True              # True: remove the line noise with a notch filter; False: leave it in the data
NOTCH_WIDTH_HZ = 4              # width of each line-noise notch (Hz), as in MNE's OPM tutorial; None: MNE's
                                # default, much narrower (the frequency / 200). Step 3 compares the choices.
                                # The notch frequencies come from the recording itself (PowerLineFrequency)
MARK_EDGES_BAD = True           # True: label the start and end of each recording BAD_edge, where the notch filter
                                # leaves some line noise (Step 3c); False: only show them. Only with APPLY_NOTCH
EDGE_S = "auto"                 # how much of each end to label, in seconds: "auto" measures it in each recording
                                # (Step 3c); or two numbers, (start, end), e.g. (1.0, 1.0)
N_JOBS = 1                      # how many CPU cores filtering and resampling may use. More can be faster;
                                # don't use more than your Jupyter session was given

# ---------------------------------------------------------------------------
# EMPTY-ROOM RECORDING (Step 5)
# ---------------------------------------------------------------------------
PROCESS_EMPTY_ROOM = True       # True: filter and downsample the session's empty-room recording the same way
                                # (Step 5); False: skip it

# ---------------------------------------------------------------------------
# SAVING (Steps 4 and 5)
# ---------------------------------------------------------------------------
OVERWRITE = True             # True: replace the saved recordings from an earlier run of this notebook

## Step 2: Find the file and load the data

This cell works out the path of the synced MEG file from your settings, checks it exists, and reads it.
If something is wrong it stops with a message that says what to fix.

`raw.load_data()` reads all the data into memory. This notebook needs that, because it filters and resamples
the whole recording. A long recording can take a minute or two to load.

Step 2b then repairs the description of the eye-tracker channels, which `read_raw_bids` loads incomplete.

In [ ]:
BIDS_ROOT = Path(BIDS_ROOT)

if "[yourpath]" in str(BIDS_ROOT):
    raise ValueError("BIDS_ROOT still contains the [yourpath] placeholder. Set it in Step 1.")

# ---- The BIDS-style name of this recording, e.g. sub-01_ses-01_task-oddball ------------------
sub_ses = [f"sub-{SUBJECT}"] + ([f"ses-{SESSION}"] if SESSION else [])   # folder levels
name_parts = sub_ses + [f"task-{TASK}"] + ([f"run-{RUN}"] if RUN else [])
base = "_".join(name_parts)        # the start of every file name for this recording
sub_dir = Path(*sub_ses)           # e.g. sub-01/ses-01

# ---- The synced MEG file from notebook 05 (MEG + eye channels) --------------------------------
eyesync_root = BIDS_ROOT / "derivatives" / EYESYNC_DERIV
# BIDSPath describes a file by its BIDS labels. Notebook 05 labels the synced file "proc-eyesync"
# (proc: a processed version of this recording).
bids_path = mne_bids.BIDSPath(root=eyesync_root, subject=SUBJECT, session=SESSION, task=TASK, run=RUN,
                              processing="eyesync", datatype="meg", suffix="meg", extension=".fif")
# FIF files can't be bigger than 2 GB, so MNE saves a longer recording in parts:
# ..._split-01_meg.fif, ..._split-02_meg.fif, ... We open the first part and MNE finds the others itself.
if not bids_path.fpath.is_file() and bids_path.copy().update(split="01").fpath.is_file():
    bids_path.update(split="01")
meg_file = bids_path.fpath
meg_dir = meg_file.parent

if not meg_file.is_file():
    found = sorted(meg_dir.glob("*_meg.fif")) if meg_dir.is_dir() else []
    raise FileNotFoundError(f"Synced MEG file not found:\n    {meg_file}\nHas notebook 05 been run for this "
                            "session? FIF files in that folder: " + (", ".join(p.name for p in found) or "none"))

# ---- The synced recording's _channels.tsv (written by notebook 05, next to the FIF file) -----------
# It lists every channel, the eye-tracking ones included, with any channels already marked bad and why.
# Notebook 01 saved an updated copy, with its bad channels, in derivatives/sensorqc/ (used in Step 2c).
bids_channels_tsv = bids_path.copy().update(split=None, suffix="channels", extension=".tsv").fpath
if not bids_channels_tsv.is_file():
    raise FileNotFoundError(f"The synced recording's _channels.tsv was not found:\n    {bids_channels_tsv}\n"
                            "Re-run notebook 05 for this session; newer versions write it.")

# ---- Read it --------------------------------------------------------------------------------
# read_raw_bids reads the FIF file together with its sidecar files in derivatives/eyesync/: the channel types
# and bad channels come from _channels.tsv, and the task and eye-tracking events from _events.tsv (they appear
# as annotations).
#
# You will see one warning here, "participants.tsv file not found". It is expected and harmless:
# read_raw_bids also looks for participants.tsv (age, sex, handedness, ...) in the top folder of the dataset
# it reads from, to copy those details into the recording's header. Here that folder is derivatives/eyesync/,
# but participants.tsv only exists at the top of the main BIDS dataset, so it isn't found and those details
# are not added. Nothing in this notebook uses them.
raw = mne_bids.read_raw_bids(bids_path)
raw.load_data()   # read the data themselves into memory (the steps below need them)

print("Loaded:", meg_file)

if len(raw.filenames) > 1:
    print(f"(the recording is split into {len(raw.filenames)} files; MNE read them all)")

# read_raw_bids loses the coil type of the eye-tracker channels (see notebook 01, Step 2b);
# the FIF file's own header still has it, so copy it back. No file is changed.
header = mne.io.read_info(meg_file, verbose=False)     # header only: quick, no data
for ch_in_file in header["chs"]:
    if ch_in_file["ch_name"] not in raw.ch_names:   # (every channel should be there; skip any that isn't)
        continue
    ch = raw.info["chs"][raw.ch_names.index(ch_in_file["ch_name"])]
    if ch["kind"] == ch_in_file["kind"]:
        ch["coil_type"] = ch_in_file["coil_type"]

types = dict(zip(raw.ch_names, raw.get_channel_types()))   # errors if the repair didn't work
print("Eye-tracker channels:", {n: t for n, t in types.items() if t in ("eyegaze", "pupil")})


In [ ]:
# Check that the recording loaded as expected:
print(raw)        # a one-line summary: number of channels, duration and size
print(raw.info)   # the header: sampling rate, filters already applied, bad channels, ...


## Step 2c: Take the bad channels and breaks from notebook 01

`read_raw_bids` set the bad channels from notebook 05's `_channels.tsv` in `derivatives/eyesync/`. That list is
from **before** the sensor-quality checks of notebook 01. Notebook 01 saved the up-to-date list, with a reason for
each channel, in `derivatives/sensorqc/` (its Step 7c). This cell replaces the bad channels with that list.

It also adds the **breaks** between blocks that notebook 01 found (its Step 7a), as `BAD_break` annotations. Like
every annotation starting with `BAD`, they are left out of later steps (for example epoching and ICA).

Both have to be in place before Step 4 saves the recording, so every later notebook gets them too.

In [ ]:
sensorqc_root = BIDS_ROOT / "derivatives" / SENSORQC_DERIV
sensorqc_dir = sensorqc_root / sub_dir / "meg"
# Notebook 01 named its files after the recording they describe, so they all start with eyesync_base
eyesync_base = f"{base}_proc-eyesync"
sensorqc_channels_tsv = sensorqc_dir / f"{eyesync_base}_channels.tsv"      # bad channels, with reasons
breaks_tsv = sensorqc_dir / f"{eyesync_base}_desc-breaks_events.tsv"      # breaks between blocks
for needed in (sensorqc_channels_tsv, breaks_tsv):
    if not needed.is_file():
        raise FileNotFoundError(f"Not found:\n    {needed}\nRun notebook 01 for this session first (its Step 7c saves it).")

# ---- Bad channels -------------------------------------------------------------------------------
sensorqc_channels = pd.read_csv(sensorqc_channels_tsv, sep="\t", dtype=str, keep_default_na=False)   # all as text
# The list must name exactly the channels of the recording: Step 4 saves it as the new recording's _channels.tsv
not_in_raw = sorted(set(sensorqc_channels["name"]) - set(raw.ch_names))
if not_in_raw:
    raise ValueError(f"{sensorqc_channels_tsv.name} lists channels that are not in the recording: {not_in_raw}")
not_in_tsv = sorted(set(raw.ch_names) - set(sensorqc_channels["name"]))
if not_in_tsv:
    raise ValueError(f"These channels of the recording are missing from {sensorqc_channels_tsv.name}: "
                     f"{not_in_tsv}\nRe-run notebook 01 for this session (its Step 7c saves the list).")
bads_from_eyesync = list(raw.info["bads"])
raw.info["bads"] = list(sensorqc_channels.loc[sensorqc_channels["status"] == "bad", "name"])

print("Bad channels from eyesync (before notebook 01):", bads_from_eyesync or "none")
print(f"Bad channels from sensorqc (used from now on): {len(raw.info['bads'])}")
display(sensorqc_channels.loc[sensorqc_channels["status"] == "bad", ["name", "status_description"]]
        .rename(columns={"name": "channel", "status_description": "reason"}).reset_index(drop=True))

# ---- Breaks -------------------------------------------------------------------------------------
# The onsets in the file count from the first sample of the recording. When the recording has a measurement date,
# MNE counts annotation onsets from that date instead, which is first_time earlier.
breaks = pd.read_csv(breaks_tsv, sep="\t")
onset_shift = raw.first_time if raw.annotations.orig_time is not None else 0.0
# Remove BAD_break annotations from an earlier run of this cell, so re-running it doesn't add them twice
raw.annotations.delete(np.flatnonzero(raw.annotations.description == "BAD_break"))
if len(breaks):
    raw.annotations.append(breaks["onset"].to_numpy() + onset_shift, breaks["duration"].to_numpy(),
                           breaks["trial_type"].to_numpy())
print(f"\n{len(breaks)} break(s) added as BAD_break:")
display(breaks)

## Step 3: Filter and downsample

Downsampling lowers the sampling rate from the recording's rate (printed in Step 2, e.g. 1200 Hz) to `TARGET_SFREQ`.
Going from 1200 Hz to the default 600 Hz halves the size of the data and makes every later step faster. This step
follows the FLUX pipeline's downsampling step.

**Why filter first.** At a sampling rate of 600 Hz, the highest frequency the data can represent is half of that,
300 Hz (the *Nyquist frequency*). Anything faster that is still in the signal doesn't disappear when you
downsample: it folds back and shows up as a fake slower signal (*aliasing*). So we first remove everything above
`LOWPASS_HZ` with a low-pass filter. Filters don't cut off sharply, so the cutoff is set well below the Nyquist
frequency: FLUX uses one third of the new sampling rate, here 600 / 3 = **200 Hz** (if you change `TARGET_SFREQ`,
`LOWPASS_HZ` follows it).

**The high-pass filter** at `HIGHPASS_HZ` (0.1 Hz) removes the slow drifts in the OPM signals at the same time.

Both are done by one **FIR filter** (MNE's default). MNE applies it forwards and backwards (*zero-phase*), so it
doesn't shift the signals in time.

**A copy, `raw_filt`.** The filtering and downsampling are done on a copy of the recording, `raw_filt`. `raw` keeps
the unfiltered data at the original rate, so Step 3b can compare the two, and you can change the filter settings in
Step 1 and run this cell again without reloading the data. From here on, `raw_filt` is the recording that gets saved.
Keeping both needs more memory: twice as much as `raw` alone while filtering (the copy is at the full rate
until it is downsampled), then about 1.5 times as much.

**Which channels.** `filter()` filters the MEG sensors and the reference sensors. The eye-tracker, trigger and
other channels are left as they are. `resample()` then changes the sampling rate of **every** channel; it
applies its own anti-aliasing to the channels that weren't filtered. For the gaze and pupil channels this means a
little smoothing: a sudden jump in them (for example at the edge of a gap that notebook 05 filled in) comes out
slightly rounded, with small ripples on either side.

**No missing values (NaN).** `resample()` works on each channel as a whole, so a single missing value (NaN) in a
channel makes that **entire** channel NaN. Notebook 05 fills in the gaps in the eye data, unless it was run with
`RESTORE_NAN = True`. This step checks every channel first and stops with a message if it finds NaN.

**Eye-tracker label channels.** Three eye-tracker channels hold labels, not signals: `ET_missing_mask` and
`ET_target_missing_mask` are 0 or 1 (1 = no data), and `ET_target_flags` holds whole-number warning codes.
`resample()` is made for smooth signals: it removes everything above the new Nyquist frequency, and a sudden step
from 0 to 1 can't stay sharp without those frequencies. It comes out rounded off, with small ripples on both sides
(values like -0.08, 0.4 or 1.08, which mean nothing for a label). So for these three channels, each new sample
takes the value of the original sample at the same time instead. Their values stay exact; their steps move by at
most one original sample (0.83 ms at 1200 Hz). Labels contain no oscillations that could fold back, so they don't
need the anti-aliasing. The trigger channels are left to `resample()`: the events come from the annotations, not
from them.

**The events stay correct.** The task and eye-tracking events were decoded by notebook 05 at the original rate
and loaded in Step 2 as **annotations**, which are stored in seconds, not samples. Resampling doesn't move
them. Two things to remember from now on:

- To get an events array, use `mne.events_from_annotations(raw)`. Don't decode the trigger channel `STI101`
  again with `mne.find_events`: resampling smears its sharp edges, so the timing would no longer be reliable.
- The `sample` column of notebook 05's `_events.tsv` counts samples at the original rate, so it no longer matches
  the data. (The `_events.tsv` that Step 4 saves has it recomputed at the new rate.)

You may see a warning that resampling the stim channels "caused event information to become unreliable". It is
expected, for the reason above: we don't use those channels for timing.

**The notch filter** (only if `APPLY_NOTCH` is True) removes the **line noise**: the magnetic field of the
building's mains power, at the **line frequency** (60 Hz in the US, 50 Hz in Europe) and its multiples
(harmonics). The line frequency is not a setting:
it is read from the recording, which has it from `PowerLineFrequency` in its `_meg.json` (`read_raw_bids` puts it
in `raw.info["line_freq"]`). The notch frequencies, `NOTCH_FREQS`, are the line frequency and its harmonics up to
the end of the low-pass's transition band: the low-pass doesn't cut off sharply, and only reaches full strength
there (250 Hz with the default 200 Hz low-pass), so a harmonic below that would partly stay. Higher harmonics are
removed by the low-pass. At 60 Hz that gives 60, 120, 180 and 240 Hz; at 50 Hz, 50 to 250 Hz, as in MNE's
[OPM tutorial](https://mne.tools/stable/auto_tutorials/preprocessing/80_opm_processing.html). Around each, the filter takes out a narrow band, `NOTCH_WIDTH_HZ` wide (4 Hz, also
as in that tutorial), and leaves the rest of the spectrum as it is. Like the band-pass, it is a zero-phase FIR filter applied to the MEG and reference
sensors, before downsampling. With `APPLY_NOTCH = False` there is no notch filter, and the line noise stays in
the data (the low-pass still removes the harmonics above it).

**Choosing the notch width.** The width is a trade-off. A wide notch also removes line noise that is spread over
a few Hz around each frequency, but it removes any brain signal in that band too: with 4 Hz at 60 Hz, everything
from 58 to 62 Hz. A narrow notch keeps more of the spectrum, but if the line-noise peak is wider than the notch,
part of it stays. The packages we looked at choose differently:

- **MNE's [OPM tutorial](https://mne.tools/stable/auto_tutorials/preprocessing/80_opm_processing.html)** (the source of the 4 Hz default here): `notch_widths=4`, at 50 to
  250 Hz.
- **MNE's default** (`notch_widths=None`): the frequency divided by 200, so 0.3 Hz at 60 Hz, 0.6 Hz at 120 Hz,
  0.9 Hz at 180 Hz and 1.2 Hz at 240 Hz ([`notch_filter`](https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.notch_filter)). MNE's
  [filtering tutorial](https://mne.tools/stable/auto_tutorials/preprocessing/30_filtering_resampling.html) uses it at 60, 120, 180 and 240 Hz. In both cases MNE adds a transition
  band of `trans_bandwidth` (default 1 Hz), half on each side of the notch.
- **The [MNE-BIDS pipeline](https://mne.tools/mne-bids-pipeline/stable/settings/preprocessing/filter.html):** no notch filter by default (`notch_freq = None`). Turned on, it
  uses MNE's default width (`notch_widths = None`).
- **[OSL](https://osl-ephys.readthedocs.io/en/latest/tutorials_build/preprocessing_manual.html):** its `notch_filter` step passes its settings on to MNE's `notch_filter`. Its tutorial
  and examples give only the frequencies (e.g. 50 and 100 Hz), so they use MNE's default width.
- **Harrison Ritz's `mne-opm` (dev branch):** uses a different method, `method="spectrum_fit"`, which doesn't cut
  out a band: it estimates the sine wave at each line frequency (in 10 s windows) and subtracts it, leaving the
  rest of the spectrum as it is. In [its pipeline settings](https://github.com/harrisonritz/mne-opm/blob/dev/docs/config-sample.py), `notch_freq = None`, with the comment that
  this skips the notch. With `spectrum_fit`, though, its pipeline (Ritz's version of the MNE-BIDS pipeline) still
  runs it: with no frequencies given, MNE looks for sine waves that stand out from the spectrum and removes those,
  and no width is involved. (MNE's [filtering tutorial](https://mne.tools/stable/auto_tutorials/preprocessing/30_filtering_resampling.html) warns that this automatic detection
  "generally does not do so reliably".) With `notch_freq` set, it removes the sine waves at those frequencies,
  within MNE's default width (the frequency / 200).

To use MNE's default width, set `NOTCH_WIDTH_HZ = None` in Step 1. The line-noise plots in Step 3b show how wide
the peaks in your recording are, and how much each notch takes out.

**The same processing for the empty room.** The steps are written once, as a function, `filter_and_downsample()`.
This cell uses it for the task recording, and Step 5 uses it for the empty-room recording, so both get exactly the
same processing.

Filtering a long recording takes a few minutes. The check in Step 3b compares a 60 s piece of `raw` and
`raw_filt`.


In [ ]:
orig_sfreq = raw.info["sfreq"]
new_nyquist = TARGET_SFREQ / 2
if TARGET_SFREQ >= orig_sfreq:
    raise ValueError(f"TARGET_SFREQ ({TARGET_SFREQ} Hz) must be lower than the recording's rate ({orig_sfreq} Hz).")
if LOWPASS_HZ >= new_nyquist:
    raise ValueError(f"LOWPASS_HZ ({LOWPASS_HZ} Hz) must be below the new Nyquist frequency ({new_nyquist} Hz).")

# ---- The line-noise frequencies for the notch filter, from the recording -------------------------------------
line_freq = raw.info["line_freq"]          # from PowerLineFrequency in the _meg.json
if APPLY_NOTCH:
    if not line_freq:
        raise ValueError("The recording has no line frequency (PowerLineFrequency in its _meg.json), so the "
                         "notch filter doesn't know which frequencies to remove.")
    # The low-pass reaches full strength at the end of its transition band. Its width is MNE's automatic one
    # (h_trans_bandwidth="auto"): this line repeats MNE's formula for it (in mne/filter.py). If a future MNE
    # changes that formula, or the filter in filter_and_downsample() is given its own h_trans_bandwidth,
    # this line has to change too, or the notch frequencies will no longer match the low-pass.
    lowpass_full_hz = LOWPASS_HZ + min(max(0.25 * LOWPASS_HZ, 2.0), orig_sfreq / 2 - LOWPASS_HZ)
    NOTCH_FREQS = np.arange(line_freq, lowpass_full_hz + 1, line_freq)   # the line frequency and its harmonics
    # The width of each notch: NOTCH_WIDTH_HZ, or MNE's default (the frequency / 200) if it is None
    NOTCH_WIDTHS = NOTCH_FREQS / 200 if NOTCH_WIDTH_HZ is None else np.full(len(NOTCH_FREQS), float(NOTCH_WIDTH_HZ))
    print(f"Line frequency {line_freq:g} Hz: notch filter at {', '.join(f'{f:g}' for f in NOTCH_FREQS)} Hz, "
          f"{', '.join(f'{w:g}' for w in NOTCH_WIDTHS)} Hz wide")
else:
    NOTCH_FREQS = np.array([])
    NOTCH_WIDTHS = np.array([])
    print("APPLY_NOTCH = False: no notch filter, the line noise stays in the data")

# The eye-tracker channels that hold labels, not signals (see the text above), if the recording has them
LABEL_CHANNELS = [ch for ch in ["ET_missing_mask", "ET_target_missing_mask", "ET_target_flags"] if ch in raw.ch_names]


def filter_and_downsample(rec, label_channels=()):
    """Filter, then downsample, a copy of the recording rec, and return the copy (rec itself is not changed).

    Used here for the task recording and in Step 5 for the empty-room recording, so both get the same processing.
    label_channels: the channels that take the nearest original sample instead (no filter; see the text above).
    """
    # ---- Stop if any channel has missing values (NaN): resampling would make the whole channel NaN ----------
    has_nan = [name for name in rec.ch_names if np.isnan(rec.get_data(picks=[name])).any()]   # one channel at a time
    if has_nan:
        raise ValueError(f"These channels contain missing values (NaN): {has_nan}\nResampling would turn each of "
                         "them into NaN from start to end. If they are eye-tracker channels, re-run notebook 05 "
                         "with RESTORE_NAN = False.")

    rec_sfreq = rec.info["sfreq"]
    print(f"Filtering {HIGHPASS_HZ:g}-{LOWPASS_HZ:g} Hz ...")
    rec_filt = rec.copy().filter(l_freq=HIGHPASS_HZ, h_freq=LOWPASS_HZ, n_jobs=N_JOBS)  # FIR, zero-phase; MEG + reference sensors
    if APPLY_NOTCH:
        print(f"Notch filter at {', '.join(f'{f:g}' for f in NOTCH_FREQS)} Hz ...")
        rec_filt.notch_filter(NOTCH_FREQS, notch_widths=NOTCH_WIDTHS, n_jobs=N_JOBS)   # the same channels as filter()
    print(f"Resampling {rec_sfreq:g} Hz -> {TARGET_SFREQ:g} Hz ...")
    rec_filt.resample(TARGET_SFREQ, n_jobs=N_JOBS)          # every channel; the annotations stay in place

    # ---- Label channels: each new sample takes the value of the original sample at the same time --------------
    nearest = np.round(rec_filt.times * rec_sfreq).astype(int)    # index of the original sample at each new time
    for ch in label_channels:
        original_values = rec.get_data(picks=[ch])[0, nearest]
        # apply_function replaces the channel's data with what the function returns: here, original_values.
        # ("values=original_values" fixes the values of this channel inside the function.)
        rec_filt.apply_function(lambda resampled, values=original_values: values, picks=[ch])
    return rec_filt


# ---- Filter, then downsample, a copy of raw -------------------------------------------------------------
# raw stays unfiltered at the original rate, so this cell can be run again with other settings.
raw_filt = filter_and_downsample(raw, LABEL_CHANNELS)
if LABEL_CHANNELS:
    print("Label channels taken from the original samples:", LABEL_CHANNELS)

print(f"\nSampling rate: {raw.info['sfreq']:g} Hz -> {raw_filt.info['sfreq']:g} Hz")
print(f"Samples:       {raw.n_times:,} -> {raw_filt.n_times:,}")
print(f"Filter stored in raw_filt.info: high-pass {raw_filt.info['highpass']:g} Hz, "
      f"low-pass {raw_filt.info['lowpass']:g} Hz")


### Step 3b: Check the result

- **Events:** the annotations of `raw_filt` must be exactly the same as those of `raw`. The table then compares the
  events MNE derives from them, in samples, in `raw` (original rate) and in `raw_filt` (new rate). At the new rate an
  event can only fall on a sample, so its time may move to the nearest one: by at most half a new sample (0.83 ms
  at 600 Hz). The line below the table checks this for every event, the eye-tracking ones included.
- **Eye-tracker label channels:** they must contain only values they had before (0 and 1 for the masks, the
  warning codes for the flags), and each value must cover the same share of the recording. The table has one row
  per value, with what it means and its share before and after; `ok` is ✓ when both hold.
- **Spectrum:** the power spectrum of the MEG sensors over the same 60 s, in `raw` (before) and `raw_filt` (after).
  Below `LOWPASS_HZ` (200 Hz with the default settings) the two curves should lie on top of each other, except
  at the notch frequencies if `APPLY_NOTCH` is True (see the last point); above it the filtered data drop away, and they end at the new Nyquist frequency (300 Hz at 600 Hz). The left plot shows the whole spectrum with frequency on a log axis, so the low
  frequencies get as much room as the high ones. The right plot zooms in on the low-pass with a linear axis, up to
  the original Nyquist frequency (600 Hz for a 1200 Hz recording), so you can see where the filtered data drop away and that nothing
  is left above 300 Hz. The slow drifts below 0.1 Hz are removed too, but the spectrum can't show
  frequencies that low; the high-pass mostly shows as traces that no longer wander off when you browse the data.
- **Line noise** (only if `APPLY_NOTCH` is True): a second figure, below the spectrum, zooms in on each frequency
  in `NOTCH_FREQS` (60, 120, 180, 240 Hz at 60 Hz), 10 Hz on either side, before and after. The shaded band is
  the notch. Before, each shows a narrow, tall peak
  (the line noise); after, the peak should be gone. The notch removes the whole shaded band, not only the peak,
  so the after curve drops deeply there and is unchanged on either side. At 240 Hz the after curve is lower
  overall, because the low-pass removes part of it too.

In [ ]:
# ---- Events -------------------------------------------------------------------------------------------
same = (len(raw_filt.annotations) == len(raw.annotations)
        and np.allclose(raw_filt.annotations.onset, raw.annotations.onset)
        and list(raw_filt.annotations.description) == list(raw.annotations.description))
print("Annotations unchanged:", same, f"({len(raw_filt.annotations)} annotations)")

# The same annotations turned into events (sample numbers) in both recordings, with the same codes
events_orig, event_id = mne.events_from_annotations(raw)
events, _ = mne.events_from_annotations(raw_filt, event_id=event_id)
if len(events) != len(events_orig) or not np.array_equal(events[:, 2], events_orig[:, 2]):
    raise RuntimeError("raw and raw_filt don't give the same list of events: re-run Step 3.")

def event_times(ev, rec):
    # sample number -> seconds from the first stored sample
    return (ev[:, 0] - rec.first_samp) / rec.info["sfreq"]

shift_ms = (event_times(events, raw_filt) - event_times(events_orig, raw)) * 1000
half_sample_ms = 1000 / (2 * raw_filt.info["sfreq"])

task_events = {k: v for k, v in event_id.items() if not k.startswith(("ET_", "BAD_"))}
task_rows = np.flatnonzero(np.isin(events[:, 2], list(task_events.values())))[:5]   # the first 5 task events
names = {v: k for k, v in event_id.items()}
sfreq_orig, sfreq_new = raw.info["sfreq"], raw_filt.info["sfreq"]
display(pd.DataFrame({
    "event": [names[c] for c in events[task_rows, 2]],
    f"sample at {sfreq_orig:g} Hz": events_orig[task_rows, 0],
    f"time at {sfreq_orig:g} Hz (s)": event_times(events_orig, raw)[task_rows],
    f"sample at {sfreq_new:g} Hz": events[task_rows, 0],
    f"time at {sfreq_new:g} Hz (s)": event_times(events, raw_filt)[task_rows],
    "shift (ms)": shift_ms[task_rows].round(3),
}))
print(f"All {len(events)} events: largest shift {np.abs(shift_ms).max():.3f} ms "
      f"(at most half a sample, {half_sample_ms:.3f} ms: {np.abs(shift_ms).max() <= half_sample_ms + 1e-6})")
print("Task events per type:", {k: int(np.sum(events[:, 2] == v)) for k, v in task_events.items()})

# ---- Eye-tracker label channels: the same values, covering the same share of the recording -------------------
# One row per value of each channel. "ok" needs: the value existed before, and its share changed by less than 0.1 %
# (steps can move by one original sample, so the shares can differ very slightly).
LABEL_MEANINGS = {   # from the descriptions in notebook 05's _channels.tsv
    "ET_missing_mask": {0: "gaze/pupil data present", 1: "no gaze/pupil data"},
    "ET_target_missing_mask": {0: "target sticker seen", 1: "no target sticker data"},
    "ET_target_flags": {-1: "no eye data", 0: "no warnings"},     # other values: EyeLink warning bits
}
label_rows = []
for ch in LABEL_CHANNELS:
    before, after = raw.get_data(picks=[ch])[0], raw_filt.get_data(picks=[ch])[0]
    values_before = np.unique(before)
    for value in np.union1d(values_before, np.unique(after)):
        pct_before, pct_after = 100 * np.mean(before == value), 100 * np.mean(after == value)
        label_rows.append({
            "channel": ch,
            "value": value,
            "meaning": LABEL_MEANINGS.get(ch, {}).get(value, "warning bits" if ch == "ET_target_flags" else "?"),
            "% before": round(pct_before, 2),
            "% after": round(pct_after, 2),
            "ok": "✓" if value in values_before and abs(pct_after - pct_before) < 0.1 else "✗",
        })
if label_rows:
    label_table = pd.DataFrame(label_rows).set_index(["channel", "value"])
    display(label_table)
    n_bad = int((label_table["ok"] == "✗").sum())
    print("Label channels:", "all values kept ✓" if n_bad == 0 else f"{n_bad} value(s) new or changed ✗ (see the table)")

# ---- Spectrum before and after ----------------------------------------------------------------------------
CHECK_TMIN, CHECK_TMAX = 60.0, 120.0      # the 60 s piece of data used for the comparison (seconds)
psd_kwargs = dict(picks="mag", exclude="bads", tmin=CHECK_TMIN, tmax=CHECK_TMAX, fmin=0.25)
# 4 s windows (0.25 Hz resolution) at each recording's own sampling rate
psd_before = raw.compute_psd(n_fft=int(4 * raw.info["sfreq"]), **psd_kwargs)
psd_after = raw_filt.compute_psd(n_fft=int(4 * raw_filt.info["sfreq"]), **psd_kwargs)

# Left: the whole spectrum, frequency on a log axis (each factor of 10 gets the same width).
# Right: zoom on the low-pass, frequency on a linear axis (each Hz gets the same width).
ZOOM_FMIN = LOWPASS_HZ / 2                 # left edge of the zoom (Hz): 100 Hz with the default settings
fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for psd, label, color in [(psd_before, f"before ({orig_sfreq:g} Hz)", "0.6"),
                          (psd_after, f"after ({raw_filt.info['sfreq']:g} Hz)", "C0")]:
    power = psd.get_data().mean(axis=0) * 1e30                       # T²/Hz -> fT²/Hz
    ax_all.plot(psd.freqs, power, color=color, label=label)
    zoom = psd.freqs >= ZOOM_FMIN
    ax_zoom.plot(psd.freqs[zoom], power[zoom], color=color, label=label)
for ax in (ax_all, ax_zoom):
    ax.axvline(LOWPASS_HZ, color="C3", ls="--", lw=1, label=f"low-pass {LOWPASS_HZ:g} Hz")
    ax.axvline(new_nyquist, color="k", ls=":", lw=1, label=f"new Nyquist {new_nyquist:g} Hz")
    ax.set(yscale="log", xlabel="Frequency (Hz)")
ax_all.set(xscale="log", ylabel="Power (fT²/Hz), mean over MEG sensors", title="Whole spectrum (log frequency axis)")
ax_zoom.set(xlim=(ZOOM_FMIN, orig_sfreq / 2), title=f"Zoom on the low-pass, {ZOOM_FMIN:g}-{orig_sfreq / 2:g} Hz (linear axis)")
ax_zoom.legend()
fig.suptitle(f"{base}: {CHECK_TMIN:g}-{CHECK_TMAX:g} s, before and after filtering + downsampling")
fig.tight_layout()
plt.show()

# ---- Line noise: the spectrum around each notch frequency, before and after ------------------------------
if APPLY_NOTCH:
    LINE_ZOOM_HZ = 10        # each plot shows this many Hz on either side of a notch frequency
    fig, axes = plt.subplots(1, len(NOTCH_FREQS), figsize=(3.5 * len(NOTCH_FREQS), 3.5), sharey=True, squeeze=False)
    for ax, freq, width in zip(axes[0], NOTCH_FREQS, NOTCH_WIDTHS):
        for psd, label, color in [(psd_before, f"before ({orig_sfreq:g} Hz)", "0.6"),
                                  (psd_after, f"after ({raw_filt.info['sfreq']:g} Hz)", "C0")]:
            near = np.abs(psd.freqs - freq) <= LINE_ZOOM_HZ
            ax.plot(psd.freqs[near], psd.get_data().mean(axis=0)[near] * 1e30, color=color, label=label)   # fT²/Hz
        ax.axvspan(freq - width / 2, freq + width / 2, color="C3", alpha=0.1)   # the notch
        ax.set(yscale="log", xlabel="Frequency (Hz)", title=f"{freq:g} Hz (notch {width:g} Hz)")
    axes[0, 0].set_ylabel("Power (fT²/Hz), mean over MEG sensors")
    axes[0, -1].legend(fontsize="small")
    fig.suptitle(f"{base}: line noise before and after the notch filter (shaded: the notch)")
    fig.tight_layout()
    plt.show()
else:
    print("APPLY_NOTCH = False: no notch filter, so the line noise (peaks at the line frequency and its "
          "harmonics) is still in the data.")

### Step 3c: The two ends of the recording (`BAD_edge`)

A filter computes each sample from the samples around it, before and after. At the very start of the recording
there are no samples before, and at the very end none after, so MNE fills in a mirror image of the data there
(`pad="reflect_limited"`). That made-up part isn't the real signal, so close to the two ends the filter works less
well. How close depends on the length of the filter: the notch filter is long, because its notches are narrow.
MNE prints its length when it designs it (with its default 1 Hz transition band, 0.5 Hz on each side of a notch,
6.6 s), so data within half of that (3.3 s) of either end can be affected.

In the sample dataset, only the notch filter does this: with the band-pass filter alone, the two ends look like the
rest of the recording. With the notch filter, some **line noise is left at the very start and end**: the fast
fluctuations there are larger than usual, at the line frequency and its harmonics. It lasts much less than 3.3 s
(0.1 s at the start and 0.3 s at the end of the sample data's empty room). This step measures it and labels
it `BAD_edge`, so later steps leave it out (like `BAD_break`, every label starting with `BAD` is left out). If it
isn't labelled, later checks can find it instead: preprocessing notebook 04 (Steps 7b and 8) then flags the first
half-second as unusual.

**How it is measured.** The first and last 3.3 s are cut into 0.1 s pieces. For each piece and each good MEG
sensor, the size of the fast fluctuations is measured: the standard deviation after removing the piece's straight-line
trend (so slow changes don't count). Each sensor's value is divided by its usual value (its median over 200 pieces
spread over the rest of the recording). The plot shows the median over the sensors (and the 90th percentile, for the
sensors most affected), at the start (left) and the end (right): 1 means as usual.

**What is labelled.** With `EDGE_S = "auto"`, each end is labelled from the end inwards, piece by piece, as long as
the median is more than 10 % above usual (`EDGE_TOLERANCE`); the first piece back within 10 % ends it. (A single
piece above 10 % further in is ordinary variation, which the plot shows too, not the filter.) If the first piece is
already within 10 %, nothing is labelled at that end. You can set the lengths
yourself instead in Step 1, e.g. `EDGE_S = (1.0, 1.0)`. With `MARK_EDGES_BAD = False`, the plot is still shown but
nothing is labelled. Without the notch filter (`APPLY_NOTCH = False`), nothing is labelled.

Step 4 saves the labels in the `_events.tsv`, and Step 5 does the same for the empty room.

In [ ]:
EDGE_STEP_S = 0.1          # the length of the pieces the ends are cut into (s)
EDGE_TOLERANCE = 1.1       # "auto": label from each end inwards while the median sensor is more than 10 % above usual


def notch_half_length_s(rec_sfreq):
    # Half the length of the notch filter: data closer than this to an end are filtered partly with mirrored data.
    # create_filter designs the same FIR band-stop filter that notch_filter uses for one notch (its default
    # trans_bandwidth of 1 Hz is split into 0.5 Hz on each side); its length doesn't depend on the frequency.
    h = mne.filter.create_filter(None, rec_sfreq, l_freq=NOTCH_FREQS[0] + NOTCH_WIDTHS[0] / 2,
                                 h_freq=NOTCH_FREQS[0] - NOTCH_WIDTHS[0] / 2,
                                 l_trans_bandwidth=0.5, h_trans_bandwidth=0.5, verbose="error")
    return len(h) / rec_sfreq / 2


def edge_profile(rec, look_s):
    # For the first and last look_s seconds, in EDGE_STEP_S pieces: each good MEG sensor's fast fluctuations
    # (standard deviation after removing the piece's straight-line trend), divided by the sensor's usual value
    # (its median over 200 pieces spread over the rest of the recording). Returns two arrays, (sensors, pieces).
    sf = rec.info["sfreq"]
    n = int(round(EDGE_STEP_S * sf))                       # samples per piece
    k = int(np.ceil(look_s / EDGE_STEP_S))                 # pieces at each end
    picks = mne.pick_types(rec.info, meg=True, ref_meg=False, exclude="bads")

    def sizes(first_samples):
        return np.column_stack([detrend(rec.get_data(picks=picks, start=s, stop=s + n), axis=1).std(axis=1)
                                for s in first_samples])

    usual = np.median(sizes(np.linspace(k * n, rec.n_times - (k + 1) * n, 200).astype(int)), axis=1, keepdims=True)
    at_start = sizes([i * n for i in range(k)]) / usual                         # piece 0 = the first 0.1 s
    at_end = sizes([rec.n_times - (i + 1) * n for i in range(k)]) / usual       # piece 0 = the last 0.1 s
    return at_start, at_end


def edge_lengths(at_start, at_end):
    # How much of each end to label (s): EDGE_S, or with "auto" from the end inwards, up to the first piece that is
    # back within EDGE_TOLERANCE of usual (a single unusual piece further in is ordinary variation, not the filter)
    if EDGE_S != "auto":
        return float(EDGE_S[0]), float(EDGE_S[1])
    lengths = []
    for ratio in (np.median(at_start, axis=0), np.median(at_end, axis=0)):
        back = np.flatnonzero(ratio <= EDGE_TOLERANCE)               # pieces at their usual size
        n_pieces = back[0] if len(back) else len(ratio)               # the pieces before the first usual one
        lengths.append(round(n_pieces * EDGE_STEP_S, 3))
    return tuple(lengths)


def set_edges(rec, lengths):
    # Replace any BAD_edge annotations by the two ends (onsets count from the measurement date, if there is one)
    rec.annotations.delete(np.flatnonzero(rec.annotations.description == "BAD_edge"))
    shift = rec.first_time if rec.annotations.orig_time is not None else 0.0
    duration = rec.n_times / rec.info["sfreq"]
    for onset, length in ((0.0, lengths[0]), (duration - lengths[1], lengths[1])):
        if length > 0:
            rec.annotations.append(onset + shift, length, "BAD_edge")


def plot_edges(rec, at_start, at_end, lengths, look_s, title):
    duration = rec.n_times / rec.info["sfreq"]
    k = at_start.shape[1]
    centres = (np.arange(k) + 0.5) * EDGE_STEP_S             # seconds from the start / before the end
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
    for ax, ratio, x, length, side in [(axes[0], at_start, centres, lengths[0], "start"),
                                        (axes[1], at_end, duration - centres, lengths[1], "end")]:
        ax.plot(x, np.median(ratio, axis=0), "o-", color="k", ms=3, label="median over the sensors")
        ax.plot(x, np.percentile(ratio, 90, axis=0), "o-", color="0.6", ms=3, label="90th percentile")
        ax.axhline(1, color="0.3", lw=0.8)
        ax.axhline(EDGE_TOLERANCE, color="tab:red", ls="--", lw=1, label=f"{EDGE_TOLERANCE:g} x usual ('auto' limit)")
        if length > 0:
            span = (0, length) if side == "start" else (duration - length, duration)
            ax.axvspan(*span, color="tab:orange", alpha=0.3, lw=0,
                       label="BAD_edge" if MARK_EDGES_BAD else "would be BAD_edge")
        ax.set(xlabel="Time (s from the start of the recording)", title=f"The {side}",
               xlim=(0, look_s) if side == "start" else (duration - look_s, duration))
    axes[0].set_ylabel("fast fluctuations\n(x their usual size)")
    # One legend with the entries of both plots (BAD_edge may be shaded at only one end)
    entries = {}
    for ax in axes:
        for handle, label in zip(*ax.get_legend_handles_labels()):
            entries.setdefault(label, handle)
    axes[1].legend(entries.values(), entries.keys(), fontsize=8, loc="upper left")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


if not APPLY_NOTCH:
    edge_look_s = None
    print("APPLY_NOTCH = False: no notch filter, so no BAD_edge (the band-pass filter alone doesn't change the ends "
          "in the sample data).")
else:
    edge_look_s = notch_half_length_s(orig_sfreq)
    print(f"Notch filter length: {2 * edge_look_s:.1f} s, so data within {edge_look_s:.1f} s of either end can be "
          "affected. Measuring ...")
    at_start, at_end = edge_profile(raw_filt, edge_look_s)
    task_edges = edge_lengths(at_start, at_end)
    if MARK_EDGES_BAD:
        set_edges(raw_filt, task_edges)
        print(f"BAD_edge: the first {task_edges[0]:g} s and the last {task_edges[1]:g} s "
              f"({'measured' if EDGE_S == 'auto' else 'set in Step 1'})")
    else:
        set_edges(raw_filt, (0.0, 0.0))
        print(f"MARK_EDGES_BAD is False: no BAD_edge. It would cover the first {task_edges[0]:g} s and the last "
              f"{task_edges[1]:g} s.")
    plot_edges(raw_filt, at_start, at_end, task_edges, edge_look_s,
               f"{base}: fast fluctuations at the two ends, after the band-pass and notch filters")

## Step 4: Save the filtered, downsampled recording

The filtered, downsampled data so far exist only in memory. This step saves them as a new recording in
`derivatives/filtered/`, next to the other derivatives. The raw BIDS dataset and `derivatives/eyesync/` are not
changed, so you can always go back and filter again with other settings.

The new recording is labelled `proc-filt`, like notebook 05's `proc-eyesync`:

```
derivatives/filtered/sub-001/ses-001/meg/sub-001_ses-001_task-oddballftcued_proc-filt_meg.fif
```

At the lower rate the recording is smaller (half the size going from 1200 to 600 Hz), so it often fits in a single
file under the 2 GB limit of a FIF file. A longer one is split into `..._split-01_meg.fif`,
`..._split-02_meg.fif`, as before.

Next to it, this step writes the BIDS sidecar files, so `read_raw_bids` can load it the way Step 2 loaded the
eyesync recording, and so the files say how they were made:

| File | What's in it |
|---|---|
| `_meg.json` | notebook 05's, with the new `SamplingFrequency` and `RecordingDuration`; **`SoftwareFilters`** records the band-pass filter, the notch filter (if `APPLY_NOTCH`) and the downsampling (from which rate, to which rate, how); **`Sources`** lists the eyesync FIF file(s) and the sensorqc files this recording was made from; **`AssociatedEmptyRoom`** still points to the raw empty-room recording here, and Step 5 changes it to the filtered one |
| `_channels.tsv` | notebook 01's (sensorqc) list, with the bad channels and their reasons, the new `sampling_frequency`, and the filter in `low_cutoff` / `high_cutoff` |
| `_events.tsv` | notebook 05's events plus the breaks (Step 2c) and the `BAD_edge` labels (Step 3c), with the `sample` column recomputed at the new rate. `read_raw_bids` builds the annotations from this file only, so the breaks must be in it |
| `_coordsystem.json`, `_channels.json`, `_events.json` | copied from eyesync (the `_events.json` describes `BAD_break` and `BAD_edge` too) |
| `dataset_description.json` | in `derivatives/filtered/`: what made this folder, and links to the `raw`, `eyesync` and `sensorqc` datasets |

If the recording already exists from an earlier run, this step stops, unless you set `OVERWRITE = True` in Step 1.
Saving a large recording to the server can take a few minutes.

In [ ]:
import json, shutil
from datetime import datetime

new_sfreq = raw_filt.info["sfreq"]
if new_sfreq != TARGET_SFREQ:
    raise RuntimeError(f"The data are at {new_sfreq:g} Hz, not TARGET_SFREQ ({TARGET_SFREQ:g} Hz). Run Step 3 first.")

filtered_root = BIDS_ROOT / "derivatives" / FILTERED_DERIV
out_dir = filtered_root / sub_dir / "meg"
out_base = f"{base}_proc-filt"                  # every file name of the new recording starts with this
out_fif = out_dir / f"{out_base}_meg.fif"

def uri(dataset, file):
    # A BIDS URI, e.g. "bids:eyesync:sub-001/ses-001/meg/..._meg.fif" ("eyesync" as named in DatasetLinks)
    return f"bids:{dataset}:{(sub_dir / 'meg' / Path(file).name).as_posix()}"

eyesync_fifs = [Path(f) for f in raw.filenames]          # the eyesync file(s) Step 2 read (raw, not raw_filt)
sources = ([uri("eyesync", f) for f in eyesync_fifs]
           + [uri("sensorqc", sensorqc_channels_tsv), uri("sensorqc", breaks_tsv)])

# ---- dataset_description.json -------------------------------------------------------------------------
raw_version = json.loads((BIDS_ROOT / "dataset_description.json").read_text()).get("BIDSVersion")
description = {
    "Name": "Filtered and downsampled OPM-MEG",
    "BIDSVersion": raw_version,
    "DatasetType": "derivative",
    "GeneratedBy": [{"Name": "02_Downsample_Filter.ipynb (Pygers-MEG)",
                     "Description": ("Band-pass and notch FIR filters" if APPLY_NOTCH else "Band-pass FIR filter")
                                    + ", then downsampling; the settings of each recording are "
                                    "in its _meg.json (SoftwareFilters)"}],
    "DatasetLinks": {"raw": "../../", "eyesync": f"../{EYESYNC_DERIV}/", "sensorqc": f"../{SENSORQC_DERIV}/"},
}
out_dir.mkdir(parents=True, exist_ok=True)
(filtered_root / "dataset_description.json").write_text(json.dumps(description, indent=4))

# ---- The FIF file(s) -----------------------------------------------------------------------------------
# split_naming="bids": if the recording is over 2 GB, the parts are named ..._proc-filt_split-01_meg.fif, ...
print(f"Saving {out_fif.name} ...")
saved = [Path(p) for p in raw_filt.save(out_fif, split_naming="bids", overwrite=OVERWRITE)]
for p in saved:
    print(f"Saved: {p.name} ({p.stat().st_size / 1e9:.2f} GB)")
# Parts from an earlier run that this run did not write (e.g. a split-02 no longer needed) would be confusing
saved_set = {p.resolve() for p in saved}
leftovers = [p for p in sorted(out_dir.glob(f"{out_base}*_meg.fif")) if p.resolve() not in saved_set]
if leftovers:
    print("These FIF files are left over from an earlier run. Delete them:", [p.name for p in leftovers])

written = []

EDGE_DESCRIPTION = ("The start or end of the recording, where the notch filter could not fully remove the line noise "
                    "(it filters partly with mirrored data there); measured and labelled by notebook 02 (Step 3c): "
                    "left out of later processing")


def edge_spans(rec):
    # The BAD_edge annotations of rec: onsets (from the first sample) and durations, in seconds
    shift = rec.first_time if rec.annotations.orig_time is not None else 0.0
    edges = rec.annotations[rec.annotations.description == "BAD_edge"]
    return edges.onset - shift, edges.duration


def software_filters(rec_sfreq, rec_filt, label_channels):
    # The SoftwareFilters entry of a _meg.json: what filter_and_downsample() (Step 3) did. Also used in Step 5.
    downsampling = {
        "OriginalSamplingFrequencyHz": rec_sfreq,
        "NewSamplingFrequencyHz": rec_filt.info["sfreq"],
        "Function": "mne.io.Raw.resample (FFT-based, with its own anti-aliasing)",
        "AppliedTo": ("all channels except the eye-tracker label channels (see LabelChannels)" if label_channels
                      else "all channels"),
    }
    if label_channels:
        downsampling["LabelChannels"] = {
            "Channels": list(label_channels),
            "Method": "each new sample takes the value of the original sample nearest in time (no filter), so "
                      "the 0/1 masks and warning codes keep their exact values",
        }
    filters = {
        "Band-pass filter": {
            "HighpassCutoffHz": HIGHPASS_HZ,
            "LowpassCutoffHz": LOWPASS_HZ,
            "FilterType": "FIR, zero-phase (MNE-Python default design: firwin, Hamming window, automatic length "
                          "and transition bandwidths)",
            "Function": "mne.io.Raw.filter",
            "AppliedTo": "MEG and reference MEG channels (eye-tracker, trigger and misc channels not filtered)",
        },
    }
    if APPLY_NOTCH:
        filters["Notch filter"] = {
            "FrequenciesHz": [float(f) for f in NOTCH_FREQS],
            "NotchWidthsHz": [float(w) for w in NOTCH_WIDTHS],   # one per frequency
            "NotchWidthSetting": ("MNE default (frequency / 200)" if NOTCH_WIDTH_HZ is None
                                  else f"{NOTCH_WIDTH_HZ:g} Hz"),
            "FilterType": "FIR, zero-phase (MNE-Python default design)",
            "Function": "mne.io.Raw.notch_filter",
            "AppliedTo": "MEG and reference MEG channels (eye-tracker, trigger and misc channels not filtered)",
        }
    filters["Downsampling"] = downsampling
    filters["MNEVersion"] = mne.__version__
    return filters


def filtered_channels_table(channels, rec_filt, label_channels):
    # A copy of a _channels.tsv table (read as text) with the new sampling rate and the filter. Also used in Step 5.
    channels = channels.copy()
    rec_nyquist = rec_filt.info["sfreq"] / 2
    channels["sampling_frequency"] = str(float(rec_filt.info["sfreq"]))
    filtered_names = [rec_filt.ch_names[k] for k in mne.pick_types(rec_filt.info, meg=True, ref_meg=True, exclude=[])]
    is_filtered = channels["name"].isin(filtered_names)
    channels.loc[is_filtered, "low_cutoff"] = str(HIGHPASS_HZ)
    channels.loc[is_filtered, "high_cutoff"] = str(LOWPASS_HZ)
    # The other channels were not filtered, but resampling removed everything above the new Nyquist frequency
    channels.loc[~is_filtered, "high_cutoff"] = str(rec_nyquist)
    # ... except the label channels, which were not filtered at all (Step 3)
    channels.loc[channels["name"].isin(label_channels), "high_cutoff"] = "n/a"
    return channels


# ---- _meg.json: notebook 05's, updated ------------------------------------------------------------------
meg_json = json.loads((meg_dir / f"{eyesync_base}_meg.json").read_text(encoding="utf-8"))
meg_json["SamplingFrequency"] = new_sfreq
meg_json["RecordingDuration"] = float(raw_filt.times[-1])
meg_json["SoftwareFilters"] = software_filters(orig_sfreq, raw_filt, LABEL_CHANNELS)
# Only the task recording has events: say how they were carried over
meg_json["SoftwareFilters"]["Downsampling"]["Note"] = (
    f"Applied after the band-pass{' and notch filters' if APPLY_NOTCH else ' filter'}. Events were not re-decoded from the trigger channels "
    "(their sharp edges are not reliable after resampling): they are carried over from the "
    "eyesync _events.tsv in seconds.")

meg_json["Sources"] = sources
meg_json["Created"] = datetime.now().isoformat(timespec="seconds")
target = out_dir / f"{out_base}_meg.json"
target.write_text(json.dumps(meg_json, indent=4, ensure_ascii=False), encoding="utf-8")
written.append(target)

# ---- _channels.tsv: notebook 01's list (bad channels + reasons), with the new rate and filter ---------------
channels = filtered_channels_table(sensorqc_channels, raw_filt, LABEL_CHANNELS)
target = out_dir / f"{out_base}_channels.tsv"
channels.to_csv(target, sep="\t", index=False)
written.append(target)

# ---- _events.tsv: notebook 05's events + the breaks, sample column at the new rate ---------------------------
events_tsv = pd.read_csv(meg_dir / f"{eyesync_base}_events.tsv", sep="\t", dtype=str, keep_default_na=False)
break_rows = pd.DataFrame({col: "n/a" for col in events_tsv.columns}, index=range(len(breaks)))
break_rows["onset"] = [f"{x:.6f}" for x in breaks["onset"]]
break_rows["duration"] = [f"{x:.6f}" for x in breaks["duration"]]
break_rows["trial_type"] = "BAD_break"
# The BAD_edge labels from Step 3c (none if it labelled nothing), with onsets from the first sample
edge_onsets, edge_durations = edge_spans(raw_filt)
edge_rows = pd.DataFrame({col: "n/a" for col in events_tsv.columns}, index=range(len(edge_onsets)))
edge_rows["onset"] = [f"{x:.6f}" for x in edge_onsets]
edge_rows["duration"] = [f"{x:.6f}" for x in edge_durations]
edge_rows["trial_type"] = "BAD_edge"
events_out = pd.concat([events_tsv, break_rows, edge_rows], ignore_index=True)
onsets = events_out["onset"].astype(float).to_numpy()
if "sample" in events_out.columns:
    events_out["sample"] = [str(int(round(x * new_sfreq))) for x in onsets]
events_out = events_out.iloc[np.argsort(onsets, kind="stable")]
target = out_dir / f"{out_base}_events.tsv"
events_out.to_csv(target, sep="\t", index=False)
written.append(target)

events_json = json.loads((meg_dir / f"{eyesync_base}_events.json").read_text(encoding="utf-8"))
events_json.setdefault("trial_type", {}).setdefault("Levels", {})["BAD_break"] = (
    "A break between blocks (from notebook 01, sensorqc): left out of later processing")
if len(edge_rows):
    events_json["trial_type"]["Levels"]["BAD_edge"] = EDGE_DESCRIPTION
if "sample" in events_json:
    events_json["sample"]["Description"] = (f"Onset of the event in samples at {new_sfreq:g} Hz, counted from "
                                            "the first stored sample (0-based).")
target = out_dir / f"{out_base}_events.json"
target.write_text(json.dumps(events_json, indent=4, ensure_ascii=False), encoding="utf-8")
written.append(target)

# ---- Copied unchanged: _channels.json and the coordinate system file(s) -------------------------------------
shutil.copyfile(meg_dir / f"{eyesync_base}_channels.json", out_dir / f"{out_base}_channels.json")
written.append(out_dir / f"{out_base}_channels.json")
subject_session = base.split("_task-")[0]          # e.g. "sub-001_ses-001"
for coord_file in sorted(meg_dir.glob(f"{subject_session}*_coordsystem.json")):
    shutil.copyfile(coord_file, out_dir / coord_file.name)
    written.append(out_dir / coord_file.name)

print(f"\nSidecars written to {out_dir}:")
for p in written:
    print("   ", p.name)

### Step 4b: Check the saved recording

This reads the saved recording back the way the next notebook will, with `read_raw_bids`, and checks against `raw_filt` that the
sampling rate, the bad channels and the annotations (events, breaks and `BAD_edge`) came through. It reads only the header
and the sidecar files, so it is quick.

In [ ]:
check_path = mne_bids.BIDSPath(root=filtered_root, subject=SUBJECT, session=SESSION, task=TASK, run=RUN,
                               processing="filt", datatype="meg", suffix="meg", extension=".fif")
if not check_path.fpath.is_file():
    check_path.update(split="01")                  # saved in parts
check = mne_bids.read_raw_bids(check_path)         # "participants.tsv file not found": expected (see Step 2)

def count(annotations):
    return pd.Series(annotations.description).value_counts().sort_index()

results = pd.DataFrame({
    "check": ["sampling rate (Hz)", "samples", "bad channels", "annotations", "BAD_break annotations",
              "BAD_edge annotations"],
    "in memory": [raw_filt.info["sfreq"], raw_filt.n_times, sorted(raw_filt.info["bads"]), len(raw_filt.annotations),
                  int(np.sum(raw_filt.annotations.description == "BAD_break")),
                  int(np.sum(raw_filt.annotations.description == "BAD_edge"))],
    "read back": [check.info["sfreq"], check.n_times, sorted(check.info["bads"]), len(check.annotations),
                  int(np.sum(check.annotations.description == "BAD_break")),
                  int(np.sum(check.annotations.description == "BAD_edge"))],
})
results["same"] = [a == b for a, b in zip(results["in memory"], results["read back"])]
display(results)
same_onsets = (len(check.annotations) == len(raw_filt.annotations)
               and np.allclose(np.sort(check.annotations.onset), np.sort(raw_filt.annotations.onset), atol=1e-3))
print("Annotation onsets the same (within 1 ms):", same_onsets)
print("\nSoftwareFilters in the saved _meg.json:")
print(json.dumps(json.loads(check_path.copy().update(split=None, extension=".json").fpath.read_text())["SoftwareFilters"],
                 indent=2))

## Step 5: The empty-room recording: the same filter and downsampling

The **empty-room recording** (`task-noise`) was made with nobody in the helmet. Later steps use it to describe the
noise, for example the **noise covariance** for source modelling. That only works if it has been processed exactly
like the task recording: the same band-pass filter, the same notch filter (if `APPLY_NOTCH`) and the same sampling rate. So this step runs the same function
as Step 3, `filter_and_downsample()`, on the empty room, and saves it next to the task recording in
`derivatives/filtered/`.

- **Which recording:** the one notebook 05 linked in the synced recording's `_meg.json` (`AssociatedEmptyRoom`),
  read from the main BIDS dataset.
- **Bad channels:** the empty room's own list from notebook 01 (its Step 13c), in `derivatives/sensorqc/`.
- **Saved as** `..._task-noise_proc-filt_meg.fif`, with its `_meg.json` (with `SoftwareFilters` and `Sources`, as in
  Step 4), `_channels.tsv` (notebook 01's list, with the new rate and filter) and `_channels.json`. It has no events,
  but its two ends are measured and labelled `BAD_edge` as in Step 3c (with its own plot), and saved in its own
  `_events.tsv` and `_events.json`.
- **The link:** the task recording's `_meg.json` in `derivatives/filtered/` then gets `AssociatedEmptyRoom` pointing
  to the filtered empty room, so `bids_path.find_empty_room()` finds it from the filtered recording.

The cell ends by checking that link: it finds the empty room from the saved task recording, reads it back, and
compares it with the one in memory.

If there is no empty-room recording, or `PROCESS_EMPTY_ROOM = False` in Step 1, this step says so and does nothing
(the task recording's `AssociatedEmptyRoom` then still points to the raw empty room, from Step 4).


In [ ]:
er_filt = None   # the filtered empty room; stays None if this step is skipped

# ---- Which empty-room recording: the one linked in the synced recording's _meg.json ----------------------------
# Notebook 05 wrote it as a BIDS URI, e.g. "bids:raw:sub-01/ses-01/meg/sub-01_ses-01_task-noise_meg.fif"
er_link = json.loads((meg_dir / f"{eyesync_base}_meg.json").read_text(encoding="utf-8")).get("AssociatedEmptyRoom")
if not PROCESS_EMPTY_ROOM:
    print("PROCESS_EMPTY_ROOM is False in Step 1: Step 5 is skipped.")
elif not er_link:
    print("The synced recording's _meg.json names no empty-room recording (AssociatedEmptyRoom): Step 5 is skipped.")
elif not er_link.startswith("bids:raw:"):
    raise ValueError(f"Unexpected AssociatedEmptyRoom in {eyesync_base}_meg.json: {er_link}\n"
                     "Expected a file in the raw BIDS dataset (bids:raw:...).")
else:
    er_file = BIDS_ROOT / er_link[len("bids:raw:"):]       # its path in the main BIDS dataset
    if not er_file.is_file():
        raise FileNotFoundError(f"The empty-room recording linked in {eyesync_base}_meg.json was not found:\n"
                                f"    {er_file}")
    # A BIDSPath from the file name, so read_raw_bids also reads its sidecar files
    er_path = mne_bids.get_bids_path_from_fname(er_file).update(root=BIDS_ROOT)
    # read_raw_bids warns "Did not find any events.tsv": expected, the empty room has no events
    er_raw = mne_bids.read_raw_bids(er_path)
    er_raw.load_data()
    print("Loaded:", er_file)
    # It gets the task recording's notch frequencies (Step 3), so its line frequency must be the same
    if APPLY_NOTCH and er_raw.info["line_freq"] != line_freq:
        raise ValueError(f"The empty room's line frequency ({er_raw.info['line_freq']} Hz) differs from the task "
                         f"recording's ({line_freq:g} Hz): check PowerLineFrequency in their _meg.json files.")

    # ---- Its bad channels: notebook 01's list for the empty room (its Step 13c) ---------------------------------
    er_sensorqc_tsv = sensorqc_dir / er_path.copy().update(split=None, suffix="channels", extension=".tsv").basename
    if not er_sensorqc_tsv.is_file():
        raise FileNotFoundError(f"Not found:\n    {er_sensorqc_tsv}\n"
                                "Run notebook 01 for this session first (its Step 13c saves it).")
    er_channels = pd.read_csv(er_sensorqc_tsv, sep="\t", dtype=str, keep_default_na=False)   # all as text
    if set(er_channels["name"]) != set(er_raw.ch_names):
        raise ValueError(f"{er_sensorqc_tsv.name} doesn't list the same channels as the empty-room recording. "
                         "Re-run notebook 01 for this session (Step 13c).")
    er_raw.info["bads"] = list(er_channels.loc[er_channels["status"] == "bad", "name"])
    print(f"Bad channels from sensorqc: {len(er_raw.info['bads'])}")

    # ---- The same filter and downsampling as the task recording (Step 3) ----------------------------------------
    er_label_channels = [ch for ch in LABEL_CHANNELS if ch in er_raw.ch_names]   # usually none: no eye data
    er_filt = filter_and_downsample(er_raw, er_label_channels)

    # ---- Its two ends: measured and labelled BAD_edge as in Step 3c -------------------------------------------
    er_edges = (0.0, 0.0)
    if APPLY_NOTCH:
        er_at_start, er_at_end = edge_profile(er_filt, edge_look_s)
        er_edges = edge_lengths(er_at_start, er_at_end)
        set_edges(er_filt, er_edges if MARK_EDGES_BAD else (0.0, 0.0))
        print(f"BAD_edge: the first {er_edges[0]:g} s and the last {er_edges[1]:g} s"
              + ("" if MARK_EDGES_BAD else " (not labelled: MARK_EDGES_BAD is False)"))
        plot_edges(er_filt, er_at_start, er_at_end, er_edges, edge_look_s,
                   "Empty room: fast fluctuations at the two ends, after the band-pass and notch filters")

    # ---- Save the FIF file(s) -----------------------------------------------------------------------------------
    # The same name with proc-filt, e.g. sub-01_ses-01_task-noise_proc-filt
    er_out_base = er_path.copy().update(split=None, processing="filt", suffix=None, extension=None).basename
    print(f"Saving {er_out_base}_meg.fif ...")
    er_saved = [Path(p) for p in er_filt.save(out_dir / f"{er_out_base}_meg.fif", split_naming="bids",
                                              overwrite=OVERWRITE)]
    for p in er_saved:
        print(f"Saved: {p.name} ({p.stat().st_size / 1e9:.2f} GB)")

    # ---- _meg.json: the raw empty room's, updated as in Step 4 --------------------------------------------------
    er_json_file = er_path.copy().update(split=None, extension=".json").fpath
    er_meg_json = json.loads(er_json_file.read_text(encoding="utf-8"))
    er_meg_json["SamplingFrequency"] = er_filt.info["sfreq"]
    er_meg_json["RecordingDuration"] = float(er_filt.times[-1])
    er_meg_json["SoftwareFilters"] = software_filters(er_raw.info["sfreq"], er_filt, er_label_channels)
    er_meg_json["Sources"] = ([f"bids:raw:{Path(f).relative_to(BIDS_ROOT).as_posix()}" for f in er_raw.filenames]
                              + [uri("sensorqc", er_sensorqc_tsv)])
    er_meg_json["Created"] = datetime.now().isoformat(timespec="seconds")
    er_written = [out_dir / f"{er_out_base}_meg.json"]
    er_written[-1].write_text(json.dumps(er_meg_json, indent=4, ensure_ascii=False), encoding="utf-8")

    # ---- _channels.tsv: notebook 01's list, with the new rate and filter; _channels.json copied -------------------
    er_written.append(out_dir / f"{er_out_base}_channels.tsv")
    filtered_channels_table(er_channels, er_filt, er_label_channels).to_csv(er_written[-1], sep="\t", index=False)
    er_channels_json = er_path.copy().update(split=None, suffix="channels", extension=".json").fpath
    if er_channels_json.is_file():
        er_written.append(out_dir / f"{er_out_base}_channels.json")
        shutil.copyfile(er_channels_json, er_written[-1])

    # ---- _events.tsv and _events.json: its BAD_edge labels (only if there are any) -----------------------------
    er_edge_onsets, er_edge_durations = edge_spans(er_filt)
    if len(er_edge_onsets):
        er_written.append(out_dir / f"{er_out_base}_events.tsv")
        pd.DataFrame({"onset": [f"{x:.6f}" for x in er_edge_onsets],
                      "duration": [f"{x:.6f}" for x in er_edge_durations],
                      "trial_type": "BAD_edge"}).to_csv(er_written[-1], sep="\t", index=False)
        er_written.append(out_dir / f"{er_out_base}_events.json")
        er_written[-1].write_text(json.dumps({
            "onset": {"Description": "Start of the span, from the first sample of the recording", "Units": "s"},
            "duration": {"Description": "Length of the span", "Units": "s"},
            "trial_type": {"Description": "Label", "Levels": {"BAD_edge": EDGE_DESCRIPTION}},
        }, indent=4, ensure_ascii=False), encoding="utf-8")
    else:
        # No labels: remove an _events.tsv/.json left over from an earlier run with labels
        for suffix in ("events.tsv", "events.json"):
            (out_dir / f"{er_out_base}_{suffix}").unlink(missing_ok=True)

    print(f"\nSidecars written to {out_dir}:")
    for p in er_written:
        print("   ", p.name)

    # ---- Link it from the task recording's _meg.json ------------------------------------------------------------
    # "bids::" + a path inside this same dataset (derivatives/filtered)
    task_meg_json_file = out_dir / f"{out_base}_meg.json"
    task_meg_json = json.loads(task_meg_json_file.read_text(encoding="utf-8"))
    task_meg_json["AssociatedEmptyRoom"] = f"bids::{(sub_dir / 'meg' / er_saved[0].name).as_posix()}"
    task_meg_json_file.write_text(json.dumps(task_meg_json, indent=4, ensure_ascii=False), encoding="utf-8")
    print(f"\nAssociatedEmptyRoom in {task_meg_json_file.name}:", task_meg_json["AssociatedEmptyRoom"])

    # ---- Check: find the empty room from the saved task recording, the way a later notebook would --------------
    er_check_path = check_path.find_empty_room(use_sidecar_only=True)   # check_path: the task recording (Step 4b)
    # read_raw_bids warns "Did not find any events.tsv" and "participants.tsv file not found": both expected
    # (see above and Step 2)
    er_check = mne_bids.read_raw_bids(er_check_path)
    er_ok = (er_check.info["sfreq"] == er_filt.info["sfreq"] and er_check.n_times == er_filt.n_times
             and sorted(er_check.info["bads"]) == sorted(er_filt.info["bads"])
             and int(np.sum(er_check.annotations.description == "BAD_edge"))
             == int(np.sum(er_filt.annotations.description == "BAD_edge")))
    print(f"Found from the task recording: {er_check_path.basename}")
    print(f"Sampling rate {er_check.info['sfreq']:g} Hz, {len(er_check.info['bads'])} bad channel(s), "
          f"{int(np.sum(er_check.annotations.description == 'BAD_edge'))} BAD_edge, the same as in memory: {er_ok}")


## Summary

This prints what this notebook did for this subject and session, in plain words. (A methods-section description of
the whole preprocessing is in the last preprocessing notebook.)


In [ ]:
# What was done, as sentences built from the settings and results above
summary = [
    f"The eye-synced recording {eyesync_base} (derivatives/{EYESYNC_DERIV}, {orig_sfreq:g} Hz, "
    f"{raw.times[-1] / 60:.1f} min) was loaded with its sidecar files.",
    f"Its bad channels ({len(raw_filt.info['bads'])}) and the breaks between blocks ({len(breaks)}, as BAD_break "
    f"annotations) were taken from notebook 01 (derivatives/{SENSORQC_DERIV}).",
    f"The MEG and reference sensors were band-pass filtered from {HIGHPASS_HZ:g} to {LOWPASS_HZ:g} Hz (zero-phase FIR "
    f"filter, MNE-Python {mne.__version__} defaults). Then every channel was downsampled to {TARGET_SFREQ:g} Hz"
    + (f", except the eye-tracker label channels ({', '.join(LABEL_CHANNELS)}), which took the value of the nearest "
       "original sample." if LABEL_CHANNELS else "."),
    (f"Before downsampling, the line noise was removed from the same sensors with a notch filter at "
     f"{', '.join(f'{f:g}' for f in NOTCH_FREQS)} Hz ("
     + ("MNE's default widths, the frequency / 200" if NOTCH_WIDTH_HZ is None else f"{NOTCH_WIDTH_HZ:g} Hz wide")
     + "; zero-phase FIR filter)."
     if APPLY_NOTCH else "No notch filter was applied: the line noise is still in the data."),
    (f"Where the notch filter could not fully remove the line noise, the first {task_edges[0]:g} s and the last "
     f"{task_edges[1]:g} s were labelled BAD_edge ({'measured' if EDGE_S == 'auto' else 'set by hand'})."
     if APPLY_NOTCH and MARK_EDGES_BAD else "No BAD_edge labels were added."),
    f"The result was saved to derivatives/{FILTERED_DERIV} as {', '.join(p.name for p in saved)}, with its sidecar "
    "files; the events were carried over in seconds, with their sample numbers recomputed at the new rate.",
]
if er_filt is not None:
    summary.append(f"The empty-room recording ({er_path.basename}) was filtered and downsampled the same way, with "
                   f"its own bad channels from notebook 01 ({len(er_filt.info['bads'])}), saved as "
                   f"{', '.join(p.name for p in er_saved)}, and linked from the task recording's _meg.json "
                   "(AssociatedEmptyRoom)."
                   + (f" Its first {er_edges[0]:g} s and last {er_edges[1]:g} s were labelled BAD_edge."
                      if APPLY_NOTCH and MARK_EDGES_BAD else ""))
else:
    summary.append("The empty-room recording was not processed (Step 5 was skipped).")
print("\n\n".join(summary))
